# 🧊 GraphShields — Notebook 3: 3D Explanation Viewer

Loads artifacts from Drive (Notebook 1's `results_notebook_1/` graph + predictions, Notebook 2's `results_notebook_2/` SHAP + GNNExplainer outputs) and renders a denser interactive 3D graph: more target transactions, more neighboring nodes, more connections. No retraining, no GNNExplainer/SHAP recomputation — neighbors beyond the ones GNNExplainer already scored are pulled directly from the real graph structure (`pyg_graph.pt`'s `edge_index`), not invented.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, base64
import numpy as np
import pandas as pd
import torch
from IPython.display import display, HTML

DRIVE_BASE  = "/content/drive/MyDrive"
ROOT        = f"{DRIVE_BASE}/GraphShields/results_final"
SHARED_ROOT = f"{DRIVE_BASE}/GraphShields/shared"

PATHS = {
    "pyg_graph":               f"{ROOT}/graph/pyg_graph.pt",
    "transaction_ids":         f"{ROOT}/embeddings/transaction_ids.csv",
    "predictions":             f"{ROOT}/predictions/hybrid_predictions.csv",
    "explanation_graph":       f"{ROOT}/explainability/gnn/explanation_graph.json",
    "transaction_explanations": f"{ROOT}/explainability/shap/transaction_explanations.csv",
    "feature_categories":      f"{SHARED_ROOT}/feature_categories.json",
}

missing = [k for k, p in PATHS.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing artifacts: {missing}")
print("All artifacts found.")


Mounted at /content/drive
All artifacts found.


In [3]:
import importlib
if not importlib.util.find_spec("torch_geometric"):
    import os; os.system("pip install torch_geometric torch_scatter torch_sparse -q")

graph_data_object = torch.load(PATHS["pyg_graph"], map_location="cpu", weights_only=False)
edge_index_np = graph_data_object.edge_index.cpu().numpy()
src_arr, dst_arr = edge_index_np[0], edge_index_np[1]

transaction_ids_df = pd.read_csv(PATHS["transaction_ids"])
node_to_tx = dict(zip(transaction_ids_df["node_idx"], transaction_ids_df["txId"]))
tx_to_node = {v: k for k, v in node_to_tx.items()}

predictions_df = pd.read_csv(PATHS["predictions"])
# keyed by txId so all lookups below use txId — avoids float-NaN node_idx misses
risk_by_txid = dict(zip(predictions_df["txId"], predictions_df["prob"]))

with open(PATHS["explanation_graph"]) as f:
    explanation_graph = json.load(f)

shap_df      = pd.read_csv(PATHS["transaction_explanations"])
shap_by_txid = shap_df.set_index("txId").to_dict(orient="index")

def shap_info(node_idx):
    txid = node_to_tx.get(node_idx)
    row  = shap_by_txid.get(txid)
    if row is None:
        return None, None, None, None
    # top_positive_factors / top_negative_factors are JSON lists of dicts
    def parse_factors(col):
        try:
            factors = json.loads(row.get(col, "[]"))
            # Group by category, summarise impact
            seen, parts = [], []
            for f in factors:
                cat = f.get("category","Other")
                if cat not in seen:
                    seen.append(cat)
                    parts.append(f"{cat} ({f.get('impact','')})")
            return "; ".join(parts) if parts else "n/a"
        except Exception:
            return "n/a"
    pos_cat = parse_factors("top_positive_factors")
    neg_cat = parse_factors("top_negative_factors")
    pos_raw = "; ".join(f"{f['feature']}:{f['impact']}"
                        for f in json.loads(row.get("top_positive_factors","[]")))
    neg_raw = "; ".join(f"{f['feature']}:{f['impact']}"
                        for f in json.loads(row.get("top_negative_factors","[]")))
    return pos_cat, neg_cat, pos_raw or "n/a", neg_raw or "n/a"

with open(PATHS["feature_categories"]) as f:
    feature_categories = json.load(f)

print(f"Graph: {graph_data_object.num_nodes:,} nodes / {graph_data_object.num_edges:,} edges")
print(f"Predictions: {len(predictions_df):,}  |  SHAP explained: {len(shap_df)}")


Graph: 203,769 nodes / 234,355 edges
Predictions: 11,184  |  SHAP explained: 25


In [4]:
TOP_N_TARGETS    = 15
MAX_NEIGHBORS    = 25
NUM_NORMAL_NODES = 10
DEFAULT_EDGE_IMP = 0.15
DEFAULT_NODE_IMP = 0.2

target_node_idx_list = (
    predictions_df.sort_values("prob", ascending=False)
    .head(TOP_N_TARGETS)["txId"]
    .map(tx_to_node).dropna().astype(int).tolist()
)

gnn_edge_importance = {
    (e["source_node_idx"], e["target_node_idx"]): e["edge_importance"]
    for e in explanation_graph["edges"]
}
gnn_node_importance = {
    n["node_idx"]: n["node_importance"]
    for n in explanation_graph["nodes"]
}

def make_node(node_idx, group, imp):
    txid = node_to_tx.get(node_idx, "unknown")
    pos_cat, neg_cat, pos_raw, neg_raw = shap_info(node_idx)
    return {
        "id": node_idx, "txId": str(txid), "group": group,
        "gnn_importance": imp,
        "predicted_risk": risk_by_txid.get(txid),
        "shap_increasing_cat": pos_cat or "n/a",
        "shap_decreasing_cat": neg_cat or "n/a",
        "shap_increasing_raw": pos_raw or "n/a",
        "shap_decreasing_raw": neg_raw or "n/a",
    }

nodes_by_id = {}
edge_rows   = []

for t_node in target_node_idx_list:
    nodes_by_id[t_node] = make_node(t_node, "target", 1.0)
    incident = np.where((src_arr == t_node) | (dst_arr == t_node))[0]
    if len(incident) > MAX_NEIGHBORS:
        scored   = [i for i in incident if (int(src_arr[i]), int(dst_arr[i])) in gnn_edge_importance]
        unscored = [i for i in incident if i not in scored]
        incident = (scored + unscored)[:MAX_NEIGHBORS]
    for e in incident:
        s, d = int(src_arr[e]), int(dst_arr[e])
        edge_rows.append({"source": s, "target": d,
                          "importance": gnn_edge_importance.get((s, d), DEFAULT_EDGE_IMP)})
        other = d if s == t_node else s
        if other in nodes_by_id and nodes_by_id[other]["group"] == "target":
            continue
        imp = max(nodes_by_id[other]["gnn_importance"], gnn_node_importance.get(other, DEFAULT_NODE_IMP))               if other in nodes_by_id else gnn_node_importance.get(other, DEFAULT_NODE_IMP)
        nodes_by_id[other] = make_node(other, "neighbor", imp)

# Normal nodes: lowest risk, not already in graph
added = 0
for _, row in predictions_df.sort_values("prob", ascending=True).iterrows():
    n_node = tx_to_node.get(row["txId"])
    if n_node is not None and n_node not in nodes_by_id and added < NUM_NORMAL_NODES:
        nodes_by_id[n_node] = make_node(n_node, "normal", DEFAULT_NODE_IMP)
        added += 1

edges_df      = pd.DataFrame(edge_rows).drop_duplicates(subset=["source","target"])
nodes_3d      = list(nodes_by_id.values())
edges_3d      = edges_df.to_dict(orient="records")
graph_data_3d = {"nodes": nodes_3d, "links": edges_3d}

print(f"3D graph: {len(nodes_3d)} nodes "
      f"({sum(1 for n in nodes_3d if n['group']=='target')} targets, "
      f"{sum(1 for n in nodes_3d if n['group']=='neighbor')} neighbors, "
      f"{sum(1 for n in nodes_3d if n['group']=='normal')} normal), "
      f"{len(edges_3d)} edges")


3D graph: 41 nodes (15 targets, 16 neighbors, 10 normal), 19 edges


In [7]:
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>GraphShields — 3D Explanation Viewer</title>
<style>
  html, body { margin: 0; padding: 0; background: #0b0e14; overflow: hidden; font-family: -apple-system, Segoe UI, Roboto, sans-serif; }
  #graph { width: 100vw; height: 100vh; }
  #panel {
    position: fixed; top: 16px; right: 16px; width: 320px;
    background: rgba(20, 24, 34, 0.92); color: #e8eaf0; border-radius: 10px;
    padding: 16px 18px; box-shadow: 0 8px 24px rgba(0,0,0,0.4);
    font-size: 13px; line-height: 1.5; display: none;
  }
  #panel h3 { margin: 0 0 8px 0; font-size: 15px; padding-right: 20px; }
  #panel .label { color: #8c93a6; font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; margin-top: 10px; }
  #panel .value { color: #e8eaf0; word-break: break-word; }
  #panel .risk-high { color: #ff6b6b; font-weight: 600; }
  #panel .risk-low { color: #69db7c; font-weight: 600; }
  #panel .raw { color: #6b7280; font-size: 10px; font-family: monospace; margin-top: 2px; }
  #panel-close {
    position: absolute; top: 12px; right: 14px; cursor: pointer; color: #8c93a6;
    font-size: 16px; line-height: 1; background: none; border: none;
  }
  #panel-close:hover { color: #e8eaf0; }
  #legend {
    position: fixed; top: 16px; left: 16px; color: #c4c9d4; font-size: 12px;
    background: rgba(20,24,34,0.8); padding: 10px 14px; border-radius: 8px;
  }
  #legend span { display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 6px; }
</style>
</head>
<body>
<div id="graph"></div>
<div id="legend">
  <div><span style="background:#ff4d4d"></span>Target transaction(s)</div>
  <div><span style="background:#4d94ff"></span>Neighbor (size/edge width = importance; brighter = GNNExplainer-scored, dim default = structural)</div>
  <div><span style="background:#8c93a6"></span>Normal transaction(s)</div>
  <div style="margin-top:6px; color:#8c93a6;">Click a node for its SHAP + GNN explanation. Click empty space to close it. Drag to rotate, scroll to zoom.</div>
</div>
<div id="panel"></div>

<script src="https://unpkg.com/three@0.149.0/build/three.min.js"></script>
<script src="https://unpkg.com/3d-force-graph@1.71.3/dist/3d-force-graph.min.js"></script>
<script>
const graphData = __GRAPH_DATA__;

const panel = document.getElementById("panel");

function fmtRisk(r) {
  if (r === null || r === undefined) return "n/a (not in test set)";
  const pct = (r * 100).toFixed(1) + "%";
  const cls = r >= 0.5 ? "risk-high" : "risk-low";
  return `<span class="${cls}">${pct}</span>`;
}

function hidePanel() {
  panel.style.display = "none";
}

function showPanel(node) {
  panel.style.display = "block";
  panel.innerHTML = `
    <button id="panel-close">\u2715</button>
    <h3>${node.group === "target" ? "\ud83c\udfaf Target Transaction" : node.group === "neighbor" ? "Neighbor Transaction" : "Normal Transaction"}</h3>
    <div class="value">txId: ${node.txId}</div>
    <div class="label">Predicted Risk (Ensemble)</div>
    <div class="value">${fmtRisk(node.predicted_risk)}</div>
    <div class="label">Node Importance</div>
    <div class="value">${node.gnn_importance.toFixed(4)}</div>
    <div class="label">SHAP — Increasing Risk</div>
    <div class="value">${node.shap_increasing_cat || "n/a"}</div>
    <div class="raw">raw: ${node.shap_increasing_raw || "n/a"}</div>
    <div class="label">SHAP — Decreasing Risk</div>
    <div class="value">${node.shap_decreasing_cat || "n/a"}</div>
    <div class="raw">raw: ${node.shap_decreasing_raw || "n/a"}</div>
  `;
  document.getElementById("panel-close").addEventListener("click", (e) => {
    e.stopPropagation();
    hidePanel();
  });
}

const graphEl = document.getElementById("graph");

const Graph = ForceGraph3D()(graphEl)
  .graphData(graphData)
  .backgroundColor("#0b0e14")
  .nodeLabel(n => `${n.group === "target" ? "TARGET" : n.group === "neighbor" ? "NEIGHBOR" : "NORMAL"} \u00b7 ${n.txId}`)
  .nodeColor(n => n.group === "target" ? "#ff4d4d" : n.group === "neighbor" ? "#4d94ff" : "#8c93a6")
  .nodeVal(n => n.group === "target" ? 12 : 3 + 9 * n.gnn_importance)
  .linkWidth(l => 0.5 + 5 * l.importance)
  .linkColor(() => "rgba(255,255,255,0.3)")
  .linkDirectionalParticles(1)
  .linkDirectionalParticleWidth(l => 1 + 2 * l.importance)
  .onNodeClick(node => {
    showPanel(node);
    const distance = 80;
    const distRatio = 1 + distance / Math.hypot(node.x, node.y, node.z);
    Graph.cameraPosition(
      { x: node.x * distRatio, y: node.y * distRatio, z: node.z * distRatio },
      node,
      800
    );
  })
  .onBackgroundClick(() => hidePanel());
</script>
</body>
</html>
"""

html_out = HTML_TEMPLATE.replace("__GRAPH_DATA__", json.dumps(graph_data_3d))

b64 = base64.b64encode(html_out.encode("utf-8")).decode("ascii")
display(HTML(f'<iframe src="data:text/html;base64,{b64}" width="100%" height="700" '
              'style="border:none; border-radius:10px;"></iframe>'))